In [0]:
%sql
-- Select your catalog
use catalog project;


In [0]:
%sql
-- create Database/Schema

create schema if not exists project.sales

In [0]:
%sql
-- create Volume
create volume if not exists project.sales.data

In [0]:
%sql
-- create table
create table if not exists project.sales.data

In [0]:
# create directory

dbutils.fs.mkdirs("/Volumes/project/sales/data/bronze")
dbutils.fs.mkdirs("/Volumes/project/sales/data/silver")
dbutils.fs.mkdirs("/Volumes/project/sales/data/gold")

True

In [0]:
raw_orders = spark.read.csv("/Volumes/project/sales/data/bronze/retail_dataset.csv",header=True,inferSchema=True)
display(raw_orders)

order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned
1001,2025-07-28,C004,Customer_4,P103,Denim Jean,Apparel,1,339.97,card,shipped,no
1002,2025-09-09,C054,Customer_54,P117,USB Cable,Accessories,1,$380.61,cash,completed,no
1003,2025-08-13,C059,Customer_59,P101,T-shirt XL,Apparel,3,278.26,card,completed,no
1004,11-Sep-2025,C013,Customer_13,P107,Charger,Accessories,3,$85.51,upi,completed,no
1005,24/08/2025,C035,Customer_35,P112,Backpack,Accessories,1,343.88,cash,cancelled,no
1006,09-16-2025,C028,Customer_28,P110,Watch,Accessories,3,$75.72,upi,cancelled,no
1007,27/07/2025,C064,Customer_64,P116,keyboard,electronics,2,315.59,upi,cancelled,no
1008,2025-08-10,C069,Customer_69,P103,Denim Jean,Apparel,null,$150.28,upi,delivered,no
1009,06/09/2025,C065,Customer_65,P109,Jeans,Apparel,-1,271.99,cash,cancelled,yes
1010,2025-09-18,C031,Customer_31,P109,jeans,Apparel,1,$47.4,card,cancelled,no


In [0]:
raw_customers = spark.read.format("json").load("/Volumes/project/sales/data/bronze/customer_dataset.json")
display(raw_customers)

age,city,customer_id,gender,loyalty_tier,signup_date
35.0,Delhi,C001,M,Gold,01-Apr-2020
39.0,Gurgaon,C002,null,silver,14/11/2023
null,Kolkata,C003,F,Silver,2022/09/13
null,BENGALURU,C004,Male,null,2017-02-21
19.0,Pune,C005,Male,PLATINUM,2022/06/06
37.0,Hyderabad,C006,f,null,27-Apr-2018
50.0,Kolkata,C007,M,null,2021/11/02
59.0,Bangalore,C008,null,Silver,17-Jun-2021
22.0,BENGALURU,C009,F,null,06-May-2020
58.0,Hyderabad,C010,M,Gold,12/12/2019


Silver_Layer_ Clean_Bronz_Data

In [2]:
from pyspark.sql.functions import *

ModuleNotFoundError: No module named 'pyspark'

In [0]:
%sql
---select * from read_files("/Volumes/project/sales/data/bronze/retail_dataset.csv")
--limit 3

In [0]:
display(raw_orders.limit(5))

order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned
1001,2025-07-28,C004,Customer_4,P103,Denim Jean,Apparel,1,339.97,card,shipped,no
1002,2025-09-09,C054,Customer_54,P117,USB Cable,Accessories,1,$380.61,cash,completed,no
1003,2025-08-13,C059,Customer_59,P101,T-shirt XL,Apparel,3,278.26,card,completed,no
1004,11-Sep-2025,C013,Customer_13,P107,Charger,Accessories,3,$85.51,upi,completed,no
1005,24/08/2025,C035,Customer_35,P112,Backpack,Accessories,1,343.88,cash,cancelled,no


In [0]:
Raw_order_count = raw_orders.count()
print(f"Total rows in raw_order: {Raw_order_count}")

Total rows in raw_order: 304


In [0]:
order_timestamp = coalesce(
         try_to_timestamp(trim(col("order_date")),lit("yyyy-MM-dd")),
         try_to_timestamp(trim(col("order_date")),lit("dd-MM-yyyy")),
         try_to_timestamp(trim(col("order_date")),lit("d-MMM-yyyy")),
         try_to_timestamp(trim(col("order_date")),lit("MM-dd-yyyy")),
         try_to_timestamp(trim(col("order_date")),lit("dd/MM/yyyy"))        
        
                        )

In [0]:
clean_orders = (raw_orders
                .withColumn("order_id",trim(col("order_id")).cast("long"))
                .withColumn("order_date",order_timestamp)
                .withColumn("customer_id",trim(col("customer_id")))
                .withColumn("customer_name",trim(col("customer_name")))
                .withColumn("category",lower(col("category")))
                .withColumn("quantity",when((trim(col("quantity")).isNull()) | (trim(col("quantity")) == lit("")),lit(1)).otherwise(trim(col("quantity"))))
                .withColumn("price_clean",translate(trim(col("price")),"$,",""))
                .withColumn("price",col("price_clean").cast("double"))
                .drop("price_clean")
                .withColumn("payment_type",lower(trim(col("payment_type"))))
                .withColumn("order_status",lower(trim(col("order_status"))))
                .withColumn("returned",lower(trim(col("returned"))))
                .withColumn("total_amount",
                            round(
                                coalesce(col("quantity"),lit(0)) *
                                coalesce(col("price"),lit(0.0)),2
                            ).cast("double"))
                                            
                )





In [0]:
clean_orders = clean_orders.dropDuplicates(["order_id","product_id"]).filter(col("order_id").isNotNull() & col("product_id").isNotNull())

In [0]:
display(clean_orders.limit(5))

order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned,total_amount
1039,2025-08-19T00:00:00.000Z,C079,Customer_79,P108,T-Shirt L,apparel,1,220.61,cash,cancelled,no,220.61
1046,2025-07-26T00:00:00.000Z,C080,Customer_80,P118,jacket,apparel,3,314.63,card,shipped,yes,943.89
1064,2025-07-20T00:00:00.000Z,C072,Customer_72,P112,Backpack,accessories,3,270.6,cash,delivered,no,811.8
1086,2025-07-29T00:00:00.000Z,C020,Customer_20,P119,Sunglasses,accessories,3,158.56,card,delivered,no,475.68
1112,2025-12-08T00:00:00.000Z,C056,Customer_56,P103,denim jean,apparel,3,186.14,cash,completed,no,558.42


In [0]:
display(clean_orders.filter(col("total_amount") < 0))

order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned,total_amount
1293,2025-08-17T00:00:00.000Z,C062,Customer_62,P111,Socks,apparel,-1,186.06,card,delivered,no,-186.06
1020,2025-09-12T00:00:00.000Z,C065,Customer_65,P108,T-Shirt L,apparel,-1,50.69,card,delivered,no,-50.69
1137,2025-08-19T00:00:00.000Z,C076,Customer_76,P106,SmartPhone,electronics,-1,435.39,card,completed,no,-435.39
1226,2025-09-06T00:00:00.000Z,C044,Customer_44,P104,Headphone,electronics,-1,86.62,card,shipped,no,-86.62
1032,2025-02-08T00:00:00.000Z,C043,Customer_43,P113,Belt,accessories,-1,197.79,upi,cancelled,no,-197.79
1125,2025-09-06T00:00:00.000Z,C001,Customer_1,P107,Charger,accessories,-1,477.25,upi,delivered,no,-477.25
1009,2025-09-06T00:00:00.000Z,C065,Customer_65,P109,Jeans,apparel,-1,271.99,cash,cancelled,yes,-271.99


In [0]:
display(clean_orders.filter(col("quantity") <0))

order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned,total_amount,total_quantity
1293,2025-08-17T00:00:00.000Z,C062,Customer_62,P111,Socks,apparel,-1,186.06,card,delivered,no,-186.06,1
1020,2025-09-12T00:00:00.000Z,C065,Customer_65,P108,T-Shirt L,apparel,-1,50.69,card,delivered,no,-50.69,1
1137,2025-08-19T00:00:00.000Z,C076,Customer_76,P106,SmartPhone,electronics,-1,435.39,card,completed,no,-435.39,1
1226,2025-09-06T00:00:00.000Z,C044,Customer_44,P104,Headphone,electronics,-1,86.62,card,shipped,no,-86.62,1
1032,2025-02-08T00:00:00.000Z,C043,Customer_43,P113,Belt,accessories,-1,197.79,upi,cancelled,no,-197.79,1
1125,2025-09-06T00:00:00.000Z,C001,Customer_1,P107,Charger,accessories,-1,477.25,upi,delivered,no,-477.25,1
1009,2025-09-06T00:00:00.000Z,C065,Customer_65,P109,Jeans,apparel,-1,271.99,cash,cancelled,yes,-271.99,1


In [0]:
clean_orders = clean_orders.drop(col("total_quantity"), col("total_amount"))
display(clean_orders.limit(5))


order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned
1039,2025-08-19T00:00:00.000Z,C079,Customer_79,P108,T-Shirt L,apparel,1,220.61,cash,cancelled,no
1046,2025-07-26T00:00:00.000Z,C080,Customer_80,P118,jacket,apparel,3,314.63,card,shipped,yes
1064,2025-07-20T00:00:00.000Z,C072,Customer_72,P112,Backpack,accessories,3,270.6,cash,delivered,no
1086,2025-07-29T00:00:00.000Z,C020,Customer_20,P119,Sunglasses,accessories,3,158.56,card,delivered,no
1112,2025-12-08T00:00:00.000Z,C056,Customer_56,P103,denim jean,apparel,3,186.14,cash,completed,no


In [0]:
clean_orders = clean_orders.withColumn("quantity", when(col("quantity") == -1, lit(1)).otherwise(col("quantity")))

In [0]:
clean_orders = clean_orders.withColumn("total_amount",(col("price"))*(col("quantity")))

In [0]:
display(clean_orders.filter(col("total_amount") < 0))

order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned,total_amount


In [0]:
clean_orders_count = clean_orders.count()
print(f"Total rows in clean_odrders: {clean_orders_count}")

Total rows in clean_odrders: 295


In [0]:
clean_orders.write.format("delta").mode("overwrite").saveAsTable("project.sales.silver_orders")

In [0]:
%sql
select * from project.sales.silver_orders

order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned,total_amount
1039,2025-08-19T00:00:00.000Z,C079,Customer_79,P108,T-Shirt L,apparel,1,220.61,cash,cancelled,no,220.61
1046,2025-07-26T00:00:00.000Z,C080,Customer_80,P118,jacket,apparel,3,314.63,card,shipped,yes,943.89
1064,2025-07-20T00:00:00.000Z,C072,Customer_72,P112,Backpack,accessories,3,270.6,cash,delivered,no,811.8000000000001
1086,2025-07-29T00:00:00.000Z,C020,Customer_20,P119,Sunglasses,accessories,3,158.56,card,delivered,no,475.68
1112,2025-12-08T00:00:00.000Z,C056,Customer_56,P103,denim jean,apparel,3,186.14,cash,completed,no,558.42
1124,2025-08-07T00:00:00.000Z,C024,Customer_24,P108,t-shirt l,apparel,3,91.87,cash,completed,no,275.61
1127,2025-09-10T00:00:00.000Z,C061,Customer_61,P117,USB Cable,accessories,3,382.12,cash,returned,yes,1146.3600000000001
1145,2025-09-14T00:00:00.000Z,C030,Customer_30,P101,T-shirt XL,apparel,3,139.19,upi,completed,no,417.57
1191,2025-07-28T00:00:00.000Z,C026,Customer_26,P102,Sneakers,shoes,2,453.01,card,shipped,no,906.02
1192,2025-07-31T00:00:00.000Z,C055,Customer_55,P112,Backpack,accessories,1,143.53,upi,delivered,no,143.53


In [0]:
%sql
select count(*) from project.sales.silver_orders

count(*)
295


Raw_customers Data set


In [0]:
bronz_customers = spark.read.json("/Volumes/project/sales/data/bronze/customer_dataset.json")
display(bronz_customers.limit(5))

age,city,customer_id,gender,loyalty_tier,signup_date
35.0,Delhi,C001,M,Gold,01-Apr-2020
39.0,Gurgaon,C002,null,silver,14/11/2023
null,Kolkata,C003,F,Silver,2022/09/13
null,BENGALURU,C004,Male,null,2017-02-21
19.0,Pune,C005,Male,PLATINUM,2022/06/06


In [0]:
bronz_customers.count()

80

In [0]:
customer_timestamp = coalesce(
         try_to_timestamp(trim(col("signup_date")),lit("yyyy-MM-dd")),
         try_to_timestamp(trim(col("signup_date")),lit("dd-MM-yyyy")),
         try_to_timestamp(trim(col("signup_date")),lit("d-MMM-yyyy")),
         try_to_timestamp(trim(col("signup_date")),lit("MM-dd-yyyy")),
         try_to_timestamp(trim(col("signup_date")),lit("dd/MM/yyyy"))        
        
                        )

In [0]:
clean_customer = (
    bronz_customers
    .withColumn("customer_id",trim(col("customer_id")))
    .withColumn("gender",lower(trim(col("gender"))))
    .withColumn("gender",when(col("gender")=="f",lit("female"))
    .when(col("gender")=="m",lit("male"))
    .otherwise(col("gender")))
    .withColumn("loyalty_tier",lower(trim(col("loyalty_tier"))))
    .withColumn("city",lower(trim(col("city"))))
    .withColumn("age",when(trim(col("age")).isNull()|(trim(col("age"))==""),lit(None))
    .otherwise(col("age")))
    .withColumn("age",when(col("age").cast("int")<0,lit(None)).otherwise(col("age").cast("int")))
    .withColumn("signup_date",customer_timestamp)           
                 
                  )

In [0]:
display(clean_customer.limit(5))

age,city,customer_id,gender,loyalty_tier,signup_date
35,delhi,C001,male,gold,2020-04-01T00:00:00.000Z
39,gurgaon,C002,null,silver,2023-11-14T00:00:00.000Z
null,kolkata,C003,female,silver,null
null,bengaluru,C004,male,null,2017-02-21T00:00:00.000Z
19,pune,C005,male,platinum,null


In [0]:
clean_customer.count()

80

In [0]:
clean_customer2 = clean_customer.dropDuplicates(["customer_id"]).filter(col("customer_id").isNotNull())

In [0]:
clean_customer2.count()

80

In [0]:
clean_customer2.write.format("delta").mode("overwrite").saveAsTable("project.sales.silver_customers")

In [0]:
%sql
select * from project.sales.silver_customers

age,city,customer_id,gender,loyalty_tier,signup_date
19,pune,C005,male,platinum,null
22,bengaluru,C009,female,null,2020-05-06T00:00:00.000Z
20,gurgaon,C024,female,silver,2018-03-18T00:00:00.000Z
44,chennai,C047,null,platinum,2020-05-19T00:00:00.000Z
67,kolkata,C070,female,gold,2016-09-22T00:00:00.000Z
24,delhi,C078,male,gold,2018-06-03T00:00:00.000Z
null,bengaluru,C004,male,null,2017-02-21T00:00:00.000Z
52,noida,C012,female,platinum,2021-03-04T00:00:00.000Z
34,null,C013,male,silver,2023-09-06T00:00:00.000Z
39,null,C030,null,bronze,null


GOLD LAYER

Two table named as
1. project.sales.silver_customers
2. project.sales.silver_orders

In [0]:
silver_customers= spark.table("project.sales.silver_customers")
display(silver_customers.limit(5))

age,city,customer_id,gender,loyalty_tier,signup_date
19,pune,C005,male,platinum,null
22,bengaluru,C009,female,null,2020-05-06T00:00:00.000Z
20,gurgaon,C024,female,silver,2018-03-18T00:00:00.000Z
44,chennai,C047,null,platinum,2020-05-19T00:00:00.000Z
67,kolkata,C070,female,gold,2016-09-22T00:00:00.000Z


In [0]:
silver_orders= spark.table("project.sales.silver_orders")
display(silver_orders.limit(5))

order_id,order_date,customer_id,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned,total_amount
1039,2025-08-19T00:00:00.000Z,C079,Customer_79,P108,T-Shirt L,apparel,1,220.61,cash,cancelled,no,220.61
1046,2025-07-26T00:00:00.000Z,C080,Customer_80,P118,jacket,apparel,3,314.63,card,shipped,yes,943.89
1064,2025-07-20T00:00:00.000Z,C072,Customer_72,P112,Backpack,accessories,3,270.6,cash,delivered,no,811.8000000000001
1086,2025-07-29T00:00:00.000Z,C020,Customer_20,P119,Sunglasses,accessories,3,158.56,card,delivered,no,475.68
1112,2025-12-08T00:00:00.000Z,C056,Customer_56,P103,denim jean,apparel,3,186.14,cash,completed,no,558.42


In [0]:
order_enriched = silver_orders.join(silver_customers,on= "customer_id", how ="left")

display(order_enriched.limit(5))

customer_id,order_id,order_date,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned,total_amount,age,city,gender,loyalty_tier,signup_date
C079,1039,2025-08-19T00:00:00.000Z,Customer_79,P108,T-Shirt L,apparel,1,220.61,cash,cancelled,no,220.61,37,hyderabad,female,platinum,2023-11-30T00:00:00.000Z
C080,1046,2025-07-26T00:00:00.000Z,Customer_80,P118,jacket,apparel,3,314.63,card,shipped,yes,943.89,51,lucknow,female,silver,2016-01-10T00:00:00.000Z
C072,1064,2025-07-20T00:00:00.000Z,Customer_72,P112,Backpack,accessories,3,270.6,cash,delivered,no,811.8000000000001,19,delhi,female,platinum,2019-06-16T00:00:00.000Z
C020,1086,2025-07-29T00:00:00.000Z,Customer_20,P119,Sunglasses,accessories,3,158.56,card,delivered,no,475.68,58,bangalore,female,platinum,2020-11-05T00:00:00.000Z
C056,1112,2025-12-08T00:00:00.000Z,Customer_56,P103,denim jean,apparel,3,186.14,cash,completed,no,558.42,68,chennai,female,gold,2016-08-25T00:00:00.000Z


In [0]:
order_enriched = (order_enriched.withColumn("order_date",to_date(col("order_date")))
                                .withColumn("Year",year(col("order_date")))
                                .withColumn("Month",month(col("order_date")))
                                .withColumn("Day",dayofmonth(col("order_date")))
                                .withColumn("Week",weekofyear(col("order_date")))
                        )

display(order_enriched)

customer_id,order_id,order_date,customer_name,product_id,product_name,category,quantity,price,payment_type,order_status,returned,total_amount,age,city,gender,loyalty_tier,signup_date,Year,Month,Day,Week
C079,1039,2025-08-19,Customer_79,P108,T-Shirt L,apparel,1,220.61,cash,cancelled,no,220.61,37,hyderabad,female,platinum,2023-11-30T00:00:00.000Z,2025,8,19,34
C080,1046,2025-07-26,Customer_80,P118,jacket,apparel,3,314.63,card,shipped,yes,943.89,51,lucknow,female,silver,2016-01-10T00:00:00.000Z,2025,7,26,30
C072,1064,2025-07-20,Customer_72,P112,Backpack,accessories,3,270.6,cash,delivered,no,811.8000000000001,19,delhi,female,platinum,2019-06-16T00:00:00.000Z,2025,7,20,29
C020,1086,2025-07-29,Customer_20,P119,Sunglasses,accessories,3,158.56,card,delivered,no,475.68,58,bangalore,female,platinum,2020-11-05T00:00:00.000Z,2025,7,29,31
C056,1112,2025-12-08,Customer_56,P103,denim jean,apparel,3,186.14,cash,completed,no,558.42,68,chennai,female,gold,2016-08-25T00:00:00.000Z,2025,12,8,50
C024,1124,2025-08-07,Customer_24,P108,t-shirt l,apparel,3,91.87,cash,completed,no,275.61,20,gurgaon,female,silver,2018-03-18T00:00:00.000Z,2025,8,7,32
C061,1127,2025-09-10,Customer_61,P117,USB Cable,accessories,3,382.12,cash,returned,yes,1146.3600000000001,65,gurgaon,male,bronze,2020-06-22T00:00:00.000Z,2025,9,10,37
C030,1145,2025-09-14,Customer_30,P101,T-shirt XL,apparel,3,139.19,upi,completed,no,417.57,39,null,null,bronze,null,2025,9,14,37
C026,1191,2025-07-28,Customer_26,P102,Sneakers,shoes,2,453.01,card,shipped,no,906.02,69,gurgaon,female,platinum,null,2025,7,28,31
C055,1192,2025-07-31,Customer_55,P112,Backpack,accessories,1,143.53,upi,delivered,no,143.53,45,hyderabad,female,platinum,2017-01-01T00:00:00.000Z,2025,7,31,31


In [0]:
gold_df = (
    order_enriched.
    groupBy("product_id","product_name","customer_id","category","Year","Month","Week","Day",
            "order_date","gender","age","city","loyalty_tier")
    
    .agg(
        sum("total_amount").alias("total_revenue"),
        sum(coalesce(col("quantity"),lit(0))).alias("total_quantity"),
        countDistinct(col("order_id")).alias("total_orders"),
        min("price").alias("min_price"),
        max("price").alias("max_price"),
        avg("price").alias("avg_price"),
        sum(when(col("returned") == "yes", 1).otherwise(0)).alias("returned_count"),
        sum(when(col("returned") == "yes", col("total_amount")).otherwise(0.0)).alias("returned_amount")
    )
 )   

display(gold_df.limit(5))



product_id,product_name,customer_id,category,Year,Month,Week,Day,order_date,gender,age,city,loyalty_tier,total_revenue,total_quantity,total_orders,min_price,max_price,avg_price,returned_count,returned_amount
P109,jeans,C031,apparel,2025,9,38,18,2025-09-18,male,34,lucknow,platinum,47.4,1,1,47.4,47.4,47.4,0,0.0
P115,Mouse,C020,electronics,2025,10,41,8,2025-10-08,female,58,bangalore,platinum,691.2,3,1,230.4,230.4,230.4,0,0.0
P112,Backpack,C068,accessories,2025,7,31,29,2025-07-29,male,40,delhi,platinum,382.7,2,1,191.35,191.35,191.35,0,0.0
P115,mouse,C030,electronics,2025,7,30,22,2025-07-22,null,39,null,bronze,261.35,1,1,261.35,261.35,261.35,0,0.0
P116,keyboard,C064,electronics,2025,7,30,27,2025-07-27,female,63,mumbai,silver,631.18,2,1,315.59,315.59,315.59,0,0.0


In [0]:
gold_df = gold_df.withColumn("avg_order_value", when(col("total_orders") > 0, col("total_revenue") / col("total_orders")).otherwise(0.0)) \
    .withColumn("avg_price_per_item", when(col("total_quantity") > 0, col("total_revenue") / col("total_quantity")).otherwise(0.0)) \
    .withColumn("refreshed_at", current_timestamp())


display(gold_df.limit(5))

product_id,product_name,customer_id,category,Year,Month,Week,Day,order_date,gender,age,city,loyalty_tier,total_revenue,total_quantity,total_orders,min_price,max_price,avg_price,returned_count,returned_amount,avg_order_value,avg_price_per_item,refreshed_at
P109,jeans,C031,apparel,2025,9,38,18,2025-09-18,male,34,lucknow,platinum,47.4,1,1,47.4,47.4,47.4,0,0.0,47.4,47.4,2026-09-26T05:42:21.870Z
P115,Mouse,C020,electronics,2025,10,41,8,2025-10-08,female,58,bangalore,platinum,691.2,3,1,230.4,230.4,230.4,0,0.0,691.2,230.4,2026-09-26T05:42:21.870Z
P112,Backpack,C068,accessories,2025,7,31,29,2025-07-29,male,40,delhi,platinum,382.7,2,1,191.35,191.35,191.35,0,0.0,382.7,191.35,2026-09-26T05:42:21.870Z
P115,mouse,C030,electronics,2025,7,30,22,2025-07-22,null,39,null,bronze,261.35,1,1,261.35,261.35,261.35,0,0.0,261.35,261.35,2026-09-26T05:42:21.870Z
P116,keyboard,C064,electronics,2025,7,30,27,2025-07-27,female,63,mumbai,silver,631.18,2,1,315.59,315.59,315.59,0,0.0,631.18,315.59,2026-09-26T05:42:21.870Z


In [0]:
gold_df.count()

294

In [0]:
display(gold_df.filter(col("total_revenue") < 0))

product_id,product_name,customer_id,category,Year,Month,Week,Day,order_date,gender,age,city,loyalty_tier,total_revenue,total_quantity,total_orders,min_price,max_price,avg_price,returned_count,returned_amount,avg_order_value,avg_price_per_item,refreshed_at


In [0]:
gold_df.write.format("delta").mode("overwrite").saveAsTable("project.sales.gold_aggregates")

In [0]:
%sql
select * from project.sales.gold_aggregates

product_id,product_name,customer_id,category,Year,Month,Week,Day,order_date,gender,age,city,loyalty_tier,total_revenue,total_quantity,total_orders,min_price,max_price,avg_price,returned_count,returned_amount,avg_order_value,avg_price_per_item,refreshed_at
P109,jeans,C031,apparel,2025,9,38,18,2025-09-18,male,34,lucknow,platinum,47.4,1,1,47.4,47.4,47.4,0,0.0,47.4,47.4,2026-09-26T05:44:24.573Z
P115,Mouse,C020,electronics,2025,10,41,8,2025-10-08,female,58,bangalore,platinum,691.2,3,1,230.4,230.4,230.4,0,0.0,691.2,230.4,2026-09-26T05:44:24.573Z
P112,Backpack,C068,accessories,2025,7,31,29,2025-07-29,male,40,delhi,platinum,382.7,2,1,191.35,191.35,191.35,0,0.0,382.7,191.35,2026-09-26T05:44:24.573Z
P115,mouse,C030,electronics,2025,7,30,22,2025-07-22,null,39,null,bronze,261.35,1,1,261.35,261.35,261.35,0,0.0,261.35,261.35,2026-09-26T05:44:24.573Z
P116,keyboard,C064,electronics,2025,7,30,27,2025-07-27,female,63,mumbai,silver,631.18,2,1,315.59,315.59,315.59,0,0.0,631.18,315.59,2026-09-26T05:44:24.573Z
P101,T-shirt XL,C012,apparel,2025,8,34,19,2025-08-19,female,52,noida,platinum,960.5699999999999,3,1,320.19,320.19,320.19,1,960.5699999999999,960.5699999999999,320.19,2026-09-26T05:44:24.573Z
P109,Jeans,C027,apparel,2025,8,35,30,2025-08-30,male,54,lucknow,null,705.51,3,1,235.17,235.17,235.17,0,0.0,705.51,235.17,2026-09-26T05:44:24.573Z
P113,Belt,C047,accessories,2025,8,33,17,2025-08-17,null,44,chennai,platinum,754.86,2,1,377.43,377.43,377.43,0,0.0,754.86,377.43,2026-09-26T05:44:24.573Z
P104,Headphone,C061,electronics,2025,2,6,8,2025-02-08,male,65,gurgaon,bronze,1053.72,3,1,351.24,351.24,351.24,0,0.0,1053.72,351.24,2026-09-26T05:44:24.573Z
P100,T-Shirt XL,C029,apparel,2025,8,34,23,2025-08-23,null,21,hyderabad,null,264.78000000000003,3,1,88.26,88.26,88.26,0,0.0,264.78000000000003,88.26,2026-09-26T05:44:24.573Z


In [0]:
%sql
select  city, round(sum(total_revenue),1) as total_revenue from project.sales.gold_aggregates
group by city
order by total_revenue desc


city,total_revenue
mumbai,29950.6
delhi,23703.6
gurgaon,22119.2
null,14985.6
noida,12204.9
lucknow,11262.3
kolkata,11097.2
hyderabad,8517.0
chennai,6424.2
pune,6023.6


In [0]:
%sql
select  city, round(avg(total_revenue),1) as total_revenue from project.sales.gold_aggregates
group by city
order by total_revenue desc

city,total_revenue
pune,602.4
mumbai,587.3
gurgaon,582.1
null,555.0
kolkata,554.9
delhi,538.7
lucknow,536.3
noida,530.6
chennai,494.2
bengaluru,473.5
